# Step 1 Phase A-10：W₂ 推定量・orientation-cluster m 感度・global 較正経路 **v1.1**
2026-09-12。**v1.0 監査（ChatGPT 2026-09-11）の全項目を反映**。v1.0 は A5 凍結前の中央帯 event と float64 selection・chunk 10000 で作られており（新チャットで設計書から起こしたことによる回帰），その結果（Q≈1.08–1.11・m=100・FWFSR 0/200）は **superseded**。
v1.1：(1) primary event を **A5 凍結の Event B = {T₁ ≤ T₁,obs ∧ T₂ ≤ T₂,obs}** に置換（pseudo は 2D 閾値）；(2) **A8b 凍結 production path**（`l24_feature231`・float32 selection・float64 evaluation・chunk 20000・BLAS threads 2・plane-folded axis）を primary，float64 selection を sensitivity 併記；(3) m 判定を 4 run・fail-closed（両 rep ペアと pooled の差 CI が 0 を含み全 run 精度 gate → m=100／m=100 不適格かつ m=10 両 rep 適格 → m=10／それ以外 unresolved），5 gate 追加，4 run の cluster 集計を保存；(4) 較正経路を 2D pseudo 閾値に修正し **one-point pathway prototype** として名称・status を分離，negative／synthetic boosted positive control・brute-force 一致を hard gate；(5) W₂ を **3 位置 max-pairwise**（A11 cache の非等価 x₀ 3 点を mock として使用）に拡張，null は replicate ごとに disjoint cluster block・3 標本 max，quantile 法固定・MC p・B／n_sub／seed 感度；(6) A9/A8/A5/A11 freeze への SHA binding・module purge＋live file SHA・official では版 hard gate・POT pin・BLAS threads 固定・CVEC ℓ-block 一定・Mx basis vs M21・A5 null file SHA・出力 SHA・atomic write・component status → `A10_VALID`・final assert。
cluster ESS 要件は superseded（precision gate = positive cluster 数＋CI 相対半幅＋5-seed 幅 CV）と明記。Phase A 調査ノート：数値は rules v1.0 draft の設計根拠。


In [ ]:
# ---- A10a: mode / environment / pinned repo / module purge + live SHA / asset binding / engine / calibration sample / A5 cross-check ----
import os, sys, json, hashlib, subprocess, platform, time, warnings, importlib
A10_MODE = globals().get('A10_MODE', 'official'); IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
MT_COMMIT = '1bdd9ea8a00891d6dc3622331f6dad5b86c16c89'                                              # A8 freeze commit (A5/A9/A11/A8 freezes, t1_engine, t2b2_bridge, t2b2_run, Step 0)
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False); BASE = '/content/drive/MyDrive/mirror_topology'; WORK = '/content/a10_work'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pot==0.9.7.post1', 'threadpoolctl'], check=True)
else: BASE = os.environ.get('A10_BASE', '/tmp/a10w/base'); WORK = os.environ.get('A10_WORK', '/tmp/a10w')
os.makedirs(WORK, exist_ok=True); MT = os.path.join(WORK, 'mt_a10'); GIT_CALLS = []
def git(*a, check=True): r = subprocess.run(['git', '-C', MT, *a], capture_output=True, text=True); GIT_CALLS.append(['git', *a, r.returncode]); assert (r.returncode == 0) or not check, (a, r.stderr[:300]); return r.stdout.strip()
if not os.path.exists(os.path.join(MT, '.git')): subprocess.run(['git', 'clone', '-q', 'https://github.com/tsujikeita/mirror-topology.git', MT], check=True)
git('fetch', '-q', 'origin', MT_COMMIT, check=False); git('checkout', '-q', '--force', MT_COMMIT); git('clean', '-fdxq')
GATES, DIAG, REC, STATUS = {}, {}, {}, {}
GATES['G_repo_commit'] = (git('rev-parse', 'HEAD') == MT_COMMIT); GATES['G_repo_origin'] = ('tsujikeita/mirror-topology' in git('remote', 'get-url', 'origin')); GATES['G_repo_clean'] = (git('status', '--porcelain', '--untracked-files=all') == '')
OUT = os.path.join(BASE, 'runs_step1_phaseA', f'a10_v1.1_{A10_MODE}'); CKPT = os.path.join(OUT, 'checkpoints'); os.makedirs(CKPT, exist_ok=True)
def sha(p, block=8 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(block), b''): h.update(b)
    return h.hexdigest()
def _jsonable(o):
    import numpy as _np
    if isinstance(o, (_np.bool_,)): return bool(o)
    if isinstance(o, (_np.integer,)): return int(o)
    if isinstance(o, (_np.floating,)): return float(o)
    if isinstance(o, _np.ndarray): return o.tolist()
    raise TypeError(f'not JSON serializable: {type(o).__name__}')
def atomic_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as fh: json.dump(obj, fh, indent=1, ensure_ascii=False, default=_jsonable); fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, path)
# module purge + live file SHA (python reuses already-imported modules)
for m in ('t1_engine', 't2b2_bridge', 't2b2_run'): sys.modules.pop(m, None)
sys.path.insert(0, MT); import t1_engine as t1, t2b2_bridge as br, t2b2_run as tr
EXP_MOD = dict(t1_engine='87bf8424073af021264b12fe312ab5255b71008bdd5fe874d164d48daf034dc8', t2b2_bridge='45107d1608d50816712f1aa452d9fa39af4adc9ec035fbe9279b264760d65872', t2b2_run='03c80f2136a8ff7ffb1077749895811ef95dd9d779c7996891d89e75545ff8db')
GATES['G_live_modules'] = all(os.path.realpath(mod.__file__).startswith(os.path.realpath(MT)) and sha(mod.__file__) == EXP_MOD[n] for n, mod in (('t1_engine', t1), ('t2b2_bridge', br), ('t2b2_run', tr)))
import numpy as np, scipy, healpy as hp, pandas as pd, ot, threadpoolctl
from scipy.spatial.transform import Rotation
from scipy.special import sph_harm_y
warnings.filterwarnings('ignore')
VERS = dict(python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__, healpy=hp.__version__, pandas=pd.__version__, pot=ot.__version__)
EXPECTED_VERS = dict(python='3.13.15', numpy='2.1.3', scipy='1.16.3', healpy='1.20.0', pot='0.9.7.post1')      # Colab official environment (A8/A11 official) + pinned POT
VERS_MISMATCH = {k: (VERS[k], v) for k, v in EXPECTED_VERS.items() if VERS[k] != v}
GATES['G_env_versions'] = (len(VERS_MISMATCH) == 0) if A10_MODE == 'official' else True             # hard gate in official; smoke records only
THREADS = 2; threadpoolctl.threadpool_limits(THREADS); THREADS_LIVE = [dict(api=i['user_api'], lib=i['internal_api'], n=i['num_threads']) for i in threadpoolctl.threadpool_info()]
# notebook identity (committed copy vs live is recorded; A10 is a Phase A investigation notebook, so head_copy may be absent)
NB_BASENAME = 'MirrorTopology_Step1_A10_v1.1.ipynb'
try: NB_HEAD = tr.source_only_sha(subprocess.run(['git', '-C', MT, 'show', f'origin/main:{NB_BASENAME}'], capture_output=True, check=True).stdout)
except Exception: NB_HEAD = None
# ---- frozen asset binding ----
P = lambda *a: os.path.join(MT, *a)
ASSETS = dict(bstack=(P('results/step1_phaseA/A5_freeze/s1_Bstack_l2_4_N16_common_v1.npz'), 'ec2d3eb501c3e00af85a505d95d7141fddb4ea23ab1da971906a5c21f80eef5f'),
              a5_null=(P('results/step1_phaseA/A5_freeze/a5_null_selection.npz'), 'd0de2cf6643ff19e134567616f243298b2cc83dd081032949984c1c6abfa2b9e'),
              a5_prov=(P('results/step1_phaseA/A5_freeze/a5_provenance.json'), '33765464cd4f4bea67e78fddb769e92ccbea38e68f6ec74778a203309971ba80'),
              a9_prov=(P('results/step1_phaseA/A9_freeze/a9_v1.2.1/a9_provenance.json'), '09c03219f43ce69368c238ab61567c3e804a45302386485820b45534539d16ef'),
              a9_manifest=(P('results/step1_phaseA/A9_freeze/freeze_manifest.json'), '4b454a157b0bdbed4a6c8856b750bb0a8951f042c56a596a4971a1bc092e2c9a'),
              a9_script=(P('results/step1_phaseA/A9_freeze/s1_phaseA9_v1.2.1.py'), '2905c036a04a04e33672a07ba464d95ae6a669c8ea7b918f17a6813e864dde2c'),
              a8_manifest=(P('results/step1_phaseA/A8_freeze/freeze_manifest.json'), '3384c9c5994a8c656ca1c868ff60156ad7e627a038bfeffe320972a6fc72e031'),
              a8b_prov=(P('results/step1_phaseA/A8_freeze/a8b/official/a8b_provenance.json'), 'a5314043e025a00ffe7ca6d89985bb1b6064e398bc9ef911de7348d601c85d6a'),
              a11_manifest=(P('results/step1_phaseA/A11_freeze/freeze_manifest.json'), '6000d7b7049dc5232f3e87daab2d4e56cb069cd15145f252ba4a548bf94df981'),
              cov_P1=(P('results/step1_phaseA/A11_freeze/official/cov_cache/cov_E7_99337bfd75deea5fe.npy'), '27e2bb589722526a40ddeca1b738f6c58edb1c8efd39672b0f7080d8de133308'),
              cov_P2=(P('results/step1_phaseA/A11_freeze/official/cov_cache/cov_E7_ebc72653fa1a10e7a.npy'), '727772a10af23118f4cb4c725af369578c1d3ac9d52d0cb641e4fc6399c29219'),
              cov_P3=(P('results/step1_phaseA/A11_freeze/official/cov_cache/cov_E7_b1862f0a867b6b3de.npy'), 'dbec1b921c3d74847095d364c04222f795c59ce4eb86686817e70d7d861d53fe'),
              step0_npz=(P('docs/step0_frozen_Bpm_v1.npz'), None), step0_csv=(P('results/step0_v0.7/step0_official_v0_7.csv'), None))
ASSET_SHA = {k: sha(p) for k, (p, e) in ASSETS.items()}; GATES['G_asset_file_sha'] = all(ASSET_SHA[k] == e for k, (p, e) in ASSETS.items() if e)
a9p = json.load(open(ASSETS['a9_prov'][0])); GATES['G_a9_binding'] = (a9p['OFFICIAL'] is True and all(a9p['gates'].values()) and len(a9p['gates']) == 30)
a8p = json.load(open(ASSETS['a8b_prov'][0])); L24 = a8p['ROUTE_DECISION']['l24']; PROD = dict(route='l24_feature231', selection_dtype='float32', evaluation_dtype='float64', sample_chunk=20000, threads=2)
GATES['G_a8b_production_spec_bound'] = (a8p['status'] == 'BENCHMARK_VALID' and all(L24[k] == v for k, v in PROD.items() if k != 'threads') and a8p['registered']['threads'] == 2)
a5p = json.load(open(ASSETS['a5_prov'][0])); GATES['G_a5_binding'] = (a5p.get('status') in ('OFFICIAL', 'VALID', 'PASS', None) and all(a5p['gates'].values()))
zB = np.load(ASSETS['bstack'][0]); Bp, Bm = np.asarray(zB['Bp_stack'], np.float64), np.asarray(zB['Bm_stack'], np.float64)
GATES['G_bstack_array_sha'] = (hashlib.sha256(np.ascontiguousarray(Bp).tobytes() + np.ascontiguousarray(Bm).tobytes()).hexdigest() == 'eb51414885785b77e9d3f7fbb25e1c9396f52e19c053d58113d50e206353a93f')
z0 = np.load(ASSETS['step0_npz'][0], allow_pickle=True); CVEC = np.asarray(z0['CVEC'], np.float64); RB = br.real_basis_lm(); LM = br.lm_full(); M21 = br.M_matrix()[0]
GATES['G_cvec_sha'] = (hashlib.sha256(np.ascontiguousarray(CVEC).tobytes()).hexdigest() == '17d85b41ee0665d88418ebf8dee794d0da0a6f219053e983ec9b0501d763c8c6')
LBLK = [(slice(0, 5), 2), (slice(5, 12), 3), (slice(12, 21), 4)]; GATES['G_cvec_lblock_constant'] = all(np.ptp(CVEC[b]) == 0 for b, l in LBLK)
GATES['G_basis_order'] = ([tuple(b) for b in z0['basis_lm']] == [(int(l), int(m), cs) for (l, m, cs) in RB] and [(int(l), int(m)) for (l, m) in LM] == [(l, m) for l in (2, 3, 4) for m in range(-l, l + 1)])
s0 = pd.read_csv(ASSETS['step0_csv'][0]); r0 = s0[s0['map'] == 'PR3_Commander']; T1o, T2o = 39.67178834527284, 259.3375006282747
GATES['G_step0_obs_bound'] = (len(r0) == 1 and float(r0.iloc[0]['Splus']) == T1o and float(r0.iloc[0]['Sminus']) == T2o)
def event_B(T1, T2, t1=T1o, t2=T2o): return (T1 <= t1) & (T2 <= t2)                                   # A5-frozen primary event
# ---- covariance systems: PR3-power-matched (primary), CT-native diagnostics recorded ----
def load_C(key):
    Mx, Cr, meta = t1.load_cov_full(ASSETS[key][0], 4); man = json.load(open(ASSETS[key][0] + '.manifest.json'))
    assert meta['cov_array_sha256'] == man['cov_array_sha256'] and man['manifest']['topology'] == 'E7' and man['manifest']['params'] == dict(LAx=1.0, LAy=0.3, L1y=1.0, L2x=0.0, L2z=1.0); return Cr, man['manifest']['x0'], meta
C_CT = {}; X0 = {}; CTM = {}
for k in ('cov_P1', 'cov_P2', 'cov_P3'): C_CT[k], X0[k], CTM[k] = load_C(k)
GATES['G_cov_positions_distinct'] = (len({tuple(np.round(v, 6)) for v in X0.values()}) == 3 and max(np.linalg.norm(C_CT[a] - C_CT[b]) / np.linalg.norm(C_CT[a]) for a in C_CT for b in C_CT if a < b) > 1e-6)
c_pr3 = CVEC[[0, 5, 12]]
def matched(C):
    c_ct = np.array([np.trace(C[b, b]) / (2 * l + 1) for b, l in LBLK]); Dm = np.diag(np.concatenate([np.repeat(np.sqrt(c_pr3[i] / c_ct[i]), 2 * l + 1) for i, (b, l) in enumerate(LBLK)])); return Dm @ C @ Dm, c_ct
C_M, c_ct1 = matched(C_CT['cov_P1']); C_ISO = np.diag(CVEC)
GATES['G_matched_power'] = np.allclose([np.trace(C_M[b, b]) / (2 * l + 1) for b, l in LBLK], c_pr3, rtol=1e-12)
def psqrt(C):
    w, V = np.linalg.eigh(C); wc = np.where(w < 1e-12 * w.max(), 0.0, w); S = V @ np.diag(np.sqrt(wc)) @ V.T
    return S, dict(lambda_min=float(w.min()), clip=int(np.sum(w < 1e-12 * w.max())), sym=float(np.linalg.norm(S - S.T) / np.linalg.norm(S)), recon=float(np.linalg.norm(S @ S.T - C) / np.linalg.norm(C)))
S_M, iM = psqrt(C_M); S_I, iI = psqrt(C_ISO); GATES['G_sqrt_hard'] = (iM['sym'] < 1e-12 and iM['recon'] < 1e-10 and iM['clip'] == 0 and iI['sym'] < 1e-12 and iI['recon'] < 1e-10)
REC['covariance'] = dict(primary_system='PR3-power-matched (morphology-only covariance experiment)', model_point='E7_b1_A x0(1)=(0.31,0.21,0.42)', c_l_CT=c_ct1.tolist(), c_l_PR3=c_pr3.tolist(), sqrt_model=iM, sqrt_iso=iI, ct_intake={k: CTM[k]['eig'] for k in CTM}, positions={k: X0[k] for k in X0})
# ---- D(R): A9/A11 quadrature construction, bound to A9 and regression-tested by direct geometry + known z-rotation ----
LMAX = 4; _NT = _NP = 2 * LMAX + 2; _xg, _wg = np.polynomial.legendre.leggauss(_NT); _th = np.arccos(_xg); _ph = 2 * np.pi * np.arange(_NP) / _NP
TH, PH = np.meshgrid(_th, _ph, indexing='ij'); WQ = (np.repeat(_wg[:, None], _NP, axis=1) * (2 * np.pi / _NP)).ravel()
DIRS = np.column_stack([np.sin(TH).ravel() * np.cos(PH).ravel(), np.sin(TH).ravel() * np.sin(PH).ravel(), np.cos(TH).ravel()])
def Yc_at(dirs):
    th, ph = hp.vec2ang(dirs); return np.array([sph_harm_y(l, m, th, ph) for (l, m) in LM])
def Ymat(dirs): return (M21.conj() @ Yc_at(dirs)).real.T
YQ = Ymat(DIRS); YQW = (YQ * WQ[:, None]).T; GATES['G_quadrature_orthonormal'] = bool(np.abs(YQW @ YQ - np.eye(21)).max() < 1e-12)
DIAG['quadrature_sha256'] = dict(dirs=hashlib.sha256(np.ascontiguousarray(DIRS).tobytes()).hexdigest(), weights=hashlib.sha256(np.ascontiguousarray(WQ).tobytes()).hexdigest(), M21=hashlib.sha256(np.ascontiguousarray(M21).tobytes()).hexdigest())
def D_of_R(Rm): return YQW @ Ymat(DIRS @ Rm)
def D_batch(Rs):
    Pp = np.einsum('qj,kji->kqi', DIRS, Rs); th, ph = hp.vec2ang(Pp.reshape(-1, 3)); Y = np.array([sph_harm_y(l, m, th, ph) for (l, m) in LM]).reshape(21, len(Rs), -1)
    Yr = np.einsum('ab,bkq->kqa', M21.conj(), Y).real; return np.einsum('aq,kqb->kab', YQW, Yr)
_rg = np.random.default_rng(20260912); Rt = Rotation.random(num=8, rng=_rg).as_matrix(); Db = D_batch(Rt); dirs_t = _rg.standard_normal((200, 3)); dirs_t /= np.linalg.norm(dirs_t, axis=1, keepdims=True)
GATES['G_D_batch_matches_single'] = bool(max(np.abs(Db[k] - D_of_R(Rt[k])).max() for k in range(8)) < 1e-12)
GATES['G_D_orthogonal'] = bool(np.abs(np.einsum('kab,kac->kbc', Db, Db) - np.eye(21)).max() < 1e-10); GATES['G_D_homomorphism'] = bool(np.abs(D_of_R(Rt[0] @ Rt[1]) - Db[0] @ Db[1]).max() < 1e-10)
xg = _rg.standard_normal(21); ag = M21.conj().T @ xg
GATES['G_D_direct_geometry'] = bool(max(np.abs((Yc_at(dirs_t).T @ (M21.conj().T @ (Db[k] @ xg))).real - (Yc_at(dirs_t @ Rt[k]).T @ ag).real).max() / np.abs((Yc_at(dirs_t @ Rt[k]).T @ ag).real).max() for k in range(8)) < 1e-10)
al = 0.7; Rz = np.array([[np.cos(al), -np.sin(al), 0], [np.sin(al), np.cos(al), 0], [0, 0, 1]]); Dz = D_of_R(Rz); okz = True
for i, (l, m, cs) in enumerate(RB):                                                                    # known z-rotation: (c,s) pairs of the same (l,m) mix by cos/sin of m*alpha, m=0 fixed
    if m == 0: okz &= abs(Dz[i, i] - 1) < 1e-10
    elif cs == 'c': j = i + 1; blk = Dz[np.ix_([i, j], [i, j])]; okz &= (np.allclose(blk, [[np.cos(m * al), -np.sin(m * al)], [np.sin(m * al), np.cos(m * al)]], atol=1e-10) or np.allclose(blk, [[np.cos(m * al), np.sin(m * al)], [-np.sin(m * al), np.cos(m * al)]], atol=1e-10))
GATES['G_D_known_z_rotation'] = bool(okz and np.abs(Dz - np.diag(np.diag(Dz))).sum() - sum(abs(Dz[i, i + 1]) + abs(Dz[i + 1, i]) for i, (l, m, cs) in enumerate(RB) if m != 0 and cs == 'c') < 1e-9)
# ---- production scan (A8b frozen): float32 selection over 3072 oriented axes, float64 evaluation at the selected axis, plane-folded axis id ----
iu = np.triu_indices(21); W = np.where(iu[0] == iu[1], 1.0, 2.0); FBp64 = (Bp[:, iu[0], iu[1]] * W).T; FBm64 = (Bm[:, iu[0], iu[1]] * W).T; FBp32 = FBp64.astype(np.float32)
_vec = np.array(hp.pix2vec(16, np.arange(3072))).T; ANTIPODE = hp.vec2pix(16, -_vec[:, 0], -_vec[:, 1], -_vec[:, 2]); PLANE = np.minimum(np.arange(3072), ANTIPODE)
GATES['G_antipode_map'] = (hashlib.sha256(np.ascontiguousarray(ANTIPODE.astype(np.int32)).tobytes()).hexdigest() == '11efe112b8f388b851b5218fa1287aafa3e32ce359f98cccb7c6649d79cc599a')
CHUNK = PROD['sample_chunk']
def scan(X, selection='float32'):
    """X (n,21) float64 -> T1, T2 (float64 evaluation at selected axis), axis, plane. selection dtype per A8b production ('float32') or sensitivity ('float64')."""
    n = len(X); T1 = np.empty(n); T2 = np.empty(n); AX = np.empty(n, np.int32)
    for a in range(0, n, CHUNK):
        Xc = X[a:a + CHUNK]; f64 = Xc[:, iu[0]] * Xc[:, iu[1]]
        ax = np.argmin((f64.astype(np.float32) @ FBp32) if selection == 'float32' else (f64 @ FBp64), axis=1).astype(np.int32)
        AX[a:a + CHUNK] = ax; T1[a:a + CHUNK] = np.einsum('ij,ji->i', f64, FBp64[:, ax]); T2[a:a + CHUNK] = np.einsum('ij,ji->i', f64, FBm64[:, ax])
    return dict(T1=T1, T2=T2, AX=AX, PL=PLANE[AX])
xt = np.random.default_rng(1).standard_normal((5, 21)); d64 = scan(xt, 'float64'); d32 = scan(xt, 'float32')
GATES['G_scan_vs_direct'] = bool(max(abs(d64['T1'][i] - min(xt[i] @ Bp[a] @ xt[i] for a in range(3072))) / abs(d64['T1'][i]) for i in range(5)) < 1e-12 and max(abs(d64['T2'][i] - xt[i] @ Bm[d64['AX'][i]] @ xt[i]) / abs(d64['T2'][i]) for i in range(5)) < 1e-12)
GATES['G_f32_selection_plane_equiv_probe'] = bool(np.all((d32['AX'] == d64['AX']) | (d32['AX'] == ANTIPODE[d64['AX']])) and np.allclose(d32['T1'], d64['T1'], rtol=1e-6) and np.allclose(d32['T2'], d64['T2'], rtol=1e-6))
# ---- generator: 1R : m z, CRN across systems ----
MASTER_SEED = 20260912; STREAM = dict(gaussian=0, rotation=1, calibration=2, pseudo=3, bootstrap=4, w2=5)
def rng_for(stream, *ids): return np.random.default_rng(np.random.SeedSequence([MASTER_SEED, 11, STREAM[stream], *[int(i) for i in ids]]))
def generate(N, m, stream_ids, S_list, selections=('float32',), chunk_clusters=2000):
    K = N // m; assert K * m == N; rr = rng_for('rotation', *stream_ids); rz = rng_for('gaussian', *stream_ids); cid = np.repeat(np.arange(K), m); t_rot = 0.0
    out = {(s, sel): dict(T1=np.empty(N), T2=np.empty(N), AX=np.empty(N, np.int32), PL=np.empty(N, np.int32)) for s in range(len(S_list)) for sel in selections}
    for k0 in range(0, K, chunk_clusters):
        k1 = min(K, k0 + chunk_clusters); t = time.perf_counter(); Rs = Rotation.random(num=k1 - k0, rng=rr).as_matrix(); Ds = D_batch(Rs); t_rot += time.perf_counter() - t; Z = rz.standard_normal((k1 - k0, m, 21)); sl = slice(k0 * m, k1 * m)
        for s, S in enumerate(S_list):
            X = np.einsum('kab,kmb->kma', Ds @ S, Z).reshape(-1, 21)
            for sel in selections:
                d = scan(X, sel)
                for kk in ('T1', 'T2', 'AX', 'PL'): out[(s, sel)][kk][sl] = d[kk]
    return out, cid, dict(K=K, m=m, rotation_seconds=t_rot)
CFG = dict(smoke=dict(N=20_000, N_CAL=20_000, N_PSEUDO=50, B_BOOT=300, B_NULL=10, N_SUB_W2=(1000, 2000), N_W2_POOL=8000, N_FIT=5000),
           official=dict(N=1_000_000, N_CAL=200_000, N_PSEUDO=200, B_BOOT=2000, B_NULL=200, N_SUB_W2=(2000, 5000), N_W2_POOL=200_000, N_FIT=20_000))[A10_MODE]
# ---- calibration sample (isotropic, stream 'calibration') + A5 map-based null cross-check (medians, P(T1<=obs), P(E_B)) ----
t0 = time.time(); calo, cal_cid, cal_info = generate(CFG['N_CAL'], 100, (0,), [S_I], ('float32', 'float64')); cal = calo[(0, 'float32')]; cal64 = calo[(0, 'float64')]
zA5 = np.load(ASSETS['a5_null'][0]); a5t1, a5t2 = zA5['T1_l2_4'], zA5['T2_l2_4']; rb5 = np.random.default_rng(np.random.SeedSequence([MASTER_SEED, 11, 9]))
ci1 = np.quantile([np.median(rb5.choice(a5t1, len(a5t1))) for _ in range(2000)], [0.025, 0.975]); ci2 = np.quantile([np.median(rb5.choice(a5t2, len(a5t2))) for _ in range(2000)], [0.025, 0.975])
def wilson(k, n, z=1.959964):
    p = k / n; c = (p + z * z / (2 * n)) / (1 + z * z / n); h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n); return (float(c - h), float(c + h))
a5_p1 = int(np.sum(a5t1 <= T1o)); a5_pB = int(np.sum(event_B(a5t1, a5t2))); w1 = wilson(a5_p1, len(a5t1)); wB = wilson(a5_pB, len(a5t1)); e_p1 = float(np.mean(cal['T1'] <= T1o)); e_pB = float(np.mean(event_B(cal['T1'], cal['T2'])))
GATES['G_iso_engine_matches_A5_null'] = bool(ci1[0] <= np.median(cal['T1']) <= ci1[1] and ci2[0] <= np.median(cal['T2']) <= ci2[1] and w1[0] <= e_p1 <= w1[1] and wB[0] <= e_pB <= wB[1])
GATES['G_cal_f32_f64_selected_output_equiv'] = bool(np.all((cal['AX'] == cal64['AX']) | (cal['AX'] == ANTIPODE[cal64['AX']])) and np.allclose(cal['T1'], cal64['T1'], rtol=1e-6) and np.allclose(cal['T2'], cal64['T2'], rtol=1e-6))
REC['calibration'] = dict(N=CFG['N_CAL'], m=100, T1_med=float(np.median(cal['T1'])), T2_med=float(np.median(cal['T2'])), P_T1_le_obs=e_p1, P_eventB=e_pB, T2_q16_q84_secondary=[float(np.quantile(cal['T2'], .16)), float(np.quantile(cal['T2'], .84))], f32_f64_axis_flip_frac=float(np.mean(cal['AX'] != cal64['AX'])),
                          A5=dict(n=len(a5t1), T1_med=float(np.median(a5t1)), T1_med_CI=ci1.tolist(), T2_med=float(np.median(a5t2)), T2_med_CI=ci2.tolist(), P_T1_le_obs=a5_p1 / len(a5t1), P_T1_le_obs_wilson=w1, P_eventB=a5_pB / len(a5t1), P_eventB_wilson=wB), seconds=time.time() - t0)
mu_c = np.array([cal['T1'].mean(), cal['T2'].mean()]); Sig_c = np.cov(np.vstack([cal['T1'], cal['T2']])); Sih = np.linalg.inv(np.linalg.cholesky(Sig_c))
np.savez_compressed(os.path.join(CKPT, 'a10a_calibration.npz'), T1=cal['T1'], T2=cal['T2'], PL=cal['PL'], cid=cal_cid, T1_f64sel=cal64['T1'], T2_f64sel=cal64['T2'])
REQ_A = ['G_repo_commit', 'G_repo_origin', 'G_repo_clean', 'G_live_modules', 'G_env_versions', 'G_asset_file_sha', 'G_a9_binding', 'G_a8b_production_spec_bound', 'G_a5_binding', 'G_bstack_array_sha', 'G_cvec_sha', 'G_cvec_lblock_constant', 'G_basis_order', 'G_step0_obs_bound', 'G_cov_positions_distinct', 'G_matched_power', 'G_sqrt_hard',
         'G_quadrature_orthonormal', 'G_D_batch_matches_single', 'G_D_orthogonal', 'G_D_homomorphism', 'G_D_direct_geometry', 'G_D_known_z_rotation', 'G_antipode_map', 'G_scan_vs_direct', 'G_f32_selection_plane_equiv_probe', 'G_iso_engine_matches_A5_null', 'G_cal_f32_f64_selected_output_equiv']
STATUS['ENGINE_VALID'] = all(GATES[k] for k in REQ_A); assert STATUS['ENGINE_VALID'], {k: GATES[k] for k in REQ_A if not GATES[k]}
print(f"A10a ENGINE_VALID | mode {A10_MODE} | MT {MT_COMMIT[:12]} | threads {THREADS_LIVE} | versions {VERS} mismatch {VERS_MISMATCH}\n   calibration: T1_med {REC['calibration']['T1_med']:.1f} (A5 {REC['calibration']['A5']['T1_med']:.1f} CI {np.round(ci1,1).tolist()}) P(T1<=obs) {e_p1:.4f} (A5 {a5_p1}/1000) P(E_B) {e_pB:.4f} (A5 {a5_pB}/1000) f32/f64 flip {REC['calibration']['f32_f64_axis_flip_frac']:.4f} ({REC['calibration']['seconds']:.0f}s)")


In [ ]:
# ---- A10b: orientation-cluster m sensitivity (Event B, production path primary, float64 selection sensitivity, 4 runs, fail-closed decision) ----
M_LIST = (10, 100); REPS = (1, 2); MS = {}; KEEP = {}
def cluster_boot_logQ(hM, hI, K, B, seed):
    rb = np.random.default_rng(np.random.SeedSequence([MASTER_SEED, 11, STREAM['bootstrap'], seed])); out = np.empty(B)
    for b in range(B):
        c = np.bincount(rb.integers(0, K, K), minlength=K); sm, si = c @ hM, c @ hI; out[b] = np.log(sm / si) if (sm > 0 and si > 0) else np.nan
    return out
def analyse(dM, dI, cid, K, tag, seed):
    eM, eI = event_B(dM['T1'], dM['T2']), event_B(dI['T1'], dI['T2']); hM, hI = np.bincount(cid, weights=eM, minlength=K), np.bincount(cid, weights=eI, minlength=K); PM, PI = float(eM.mean()), float(eI.mean())
    Q = PM / PI if PI > 0 else np.inf; lq = float(np.log(Q)) if np.isfinite(Q) and Q > 0 else None
    boots = [cluster_boot_logQ(hM, hI, K, CFG['B_BOOT'], seed * 10 + s) for s in range(5)]; cis = [np.nanquantile(b, [0.025, 0.975]) if np.isfinite(b).sum() > 10 else np.array([np.nan, np.nan]) for b in boots]
    widths = np.array([c[1] - c[0] for c in cis]); ci = cis[0]; se = float(np.nanstd(boots[0])); relh = float((np.exp(ci[1]) - np.exp(ci[0])) / 2 / Q) if lq is not None and np.all(np.isfinite(ci)) else np.inf
    return dict(tag=tag, K=K, P_M=PM, P_I=PI, hits_M=int(eM.sum()), hits_I=int(eI.sum()), Q=float(Q) if np.isfinite(Q) else None, logQ=lq, se_logQ=se, logCI95=ci.tolist(), CI95=np.exp(ci).tolist(), CI_rel_halfwidth=relh, CI_width_CV_5seeds=float(widths.std() / widths.mean()) if np.all(np.isfinite(widths)) else None,
                event_positive_clusters_M=int((hM > 0).sum()), event_positive_clusters_I=int((hI > 0).sum()), precision_gate=bool((hM > 0).sum() >= 50 and (hI > 0).sum() >= 50 and relh <= 0.20 and np.all(np.isfinite(widths)) and widths.std() / widths.mean() < 0.2),
                P_M_T1_le_obs=float(np.mean(dM['T1'] <= T1o)), P_I_T1_le_obs=float(np.mean(dI['T1'] <= T1o)), T1_med_M=float(np.median(dM['T1'])), T1_med_I=float(np.median(dI['T1'])), T2_med_M=float(np.median(dM['T2'])), T2_med_I=float(np.median(dI['T2']))), hM, hI, boots[0]
BOOTS = {}
for m in M_LIST:
    for rep in REPS:
        t = time.time(); out, cid, gi = generate(CFG['N'], m, (m, rep), [S_M, S_I], ('float32', 'float64')); K = gi['K']
        r, hM, hI, bt = analyse(out[(0, 'float32')], out[(1, 'float32')], cid, K, f'm{m}_rep{rep}', 100 * m + rep); r64, hM64, hI64, _ = analyse(out[(0, 'float64')], out[(1, 'float64')], cid, K, f'm{m}_rep{rep}_f64sel', 100 * m + rep)
        r.update(rotation_seconds=gi['rotation_seconds'], seconds=time.time() - t, sensitivity_float64_selection=dict(Q=r64['Q'], logQ=r64['logQ'], CI95=r64['CI95'], hits_M=r64['hits_M'], hits_I=r64['hits_I'], axis_flip_frac_M=float(np.mean(out[(0, 'float32')]['AX'] != out[(0, 'float64')]['AX'])), all_flips_antipodal=bool(np.all((out[(0, 'float32')]['AX'] == out[(0, 'float64')]['AX']) | (out[(0, 'float32')]['AX'] == ANTIPODE[out[(0, 'float64')]['AX']]))), eventB_identical=bool(np.array_equal(event_B(out[(0,'float32')]['T1'], out[(0,'float32')]['T2']), event_B(out[(0,'float64')]['T1'], out[(0,'float64')]['T2'])))))
        MS[r['tag']] = r; BOOTS[r['tag']] = bt; KEEP[(m, rep)] = dict(hM=hM, hI=hI, hM64=hM64, hI64=hI64, K=K)
        if rep == 1: KEEP[m] = dict(dM=out[(0, 'float32')], dI=out[(1, 'float32')], cid=cid, K=K)
        print(f"{r['tag']}: K={K} P_M={r['P_M']:.5f} P_I={r['P_I']:.5f} Q={r['Q']} CI={np.round(r['CI95'],3).tolist()} relh={r['CI_rel_halfwidth']:.3f} ev+ M/I={r['event_positive_clusters_M']}/{r['event_positive_clusters_I']} prec={r['precision_gate']} | f64sel Q={r64['Q']} | {time.time()-t:.0f}s")
np.savez_compressed(os.path.join(CKPT, 'a10b_cluster_hits.npz'), **{f'm{m}_rep{rep}_{k}': KEEP[(m, rep)][k] for m in M_LIST for rep in REPS for k in ('hM', 'hI', 'hM64', 'hI64')}, **{f'{t}_boot_logQ': b for t, b in BOOTS.items()})
# decision (fail-closed, all four runs): difference CI per replicate pair and pooled; both from INDEPENDENT bootstrap draws of logQ
def diff_ci(a, b): d = BOOTS[a] - BOOTS[b]; return [float(np.nanquantile(d, 0.025)), float(np.nanquantile(d, 0.975))]
d1, d2 = diff_ci('m100_rep1', 'm10_rep1'), diff_ci('m100_rep2', 'm10_rep2'); dp = 0.5 * ((BOOTS['m100_rep1'] + BOOTS['m100_rep2']) - (BOOTS['m10_rep1'] + BOOTS['m10_rep2'])); dpci = [float(np.nanquantile(dp, 0.025)), float(np.nanquantile(dp, 0.975))]
allQ = all(MS[t]['logQ'] is not None for t in MS); prec = {t: MS[t]['precision_gate'] for t in MS}; m100_ok = prec['m100_rep1'] and prec['m100_rep2']; m10_ok = prec['m10_rep1'] and prec['m10_rep2']
zero_in = lambda c: (c[0] <= 0 <= c[1]); consistent = zero_in(d1) and zero_in(d2) and zero_in(dpci)
adopted = 100 if (allQ and m100_ok and m10_ok and consistent) else (10 if (allQ and m10_ok and not m100_ok) else None)
M_DECISION = dict(rule='m=100 iff all four runs pass the precision gate AND the m100-m10 logQ difference CI contains 0 for rep1 pair, rep2 pair and the 2-rep pooled difference; m=10 iff m100 fails and both m10 reps pass; otherwise unresolved (fail-closed)',
                  diff_logCI_rep1=d1, diff_logCI_rep2=d2, diff_logCI_pooled=dpci, point_diff_rep1=MS['m100_rep1']['logQ'] - MS['m10_rep1']['logQ'] if allQ else None, point_diff_rep2=MS['m100_rep2']['logQ'] - MS['m10_rep2']['logQ'] if allQ else None,
                  replicate_delta_logQ_m10=abs(MS['m10_rep2']['logQ'] - MS['m10_rep1']['logQ']) if allQ else None, replicate_delta_logQ_m100=abs(MS['m100_rep2']['logQ'] - MS['m100_rep1']['logQ']) if allQ else None,
                  precision=prec, adopted_m=adopted, cluster_ESS='superseded: precision gate = event-positive clusters >= 50 (both systems) AND CI relative half-width <= 0.20 AND 5-seed CI-width CV < 0.2')
GATES['G_m_run_inventory'] = (set(MS) == {f'm{m}_rep{r}' for m in M_LIST for r in REPS}); GATES['G_m_all_Q_finite'] = allQ; GATES['G_m_precision_each_run'] = all(prec.values())
GATES['G_m_two_rep_consistency'] = bool(allQ and abs(MS['m10_rep2']['logQ'] - MS['m10_rep1']['logQ']) <= 1.96 * np.sqrt(MS['m10_rep1']['se_logQ'] ** 2 + MS['m10_rep2']['se_logQ'] ** 2) and abs(MS['m100_rep2']['logQ'] - MS['m100_rep1']['logQ']) <= 1.96 * np.sqrt(MS['m100_rep1']['se_logQ'] ** 2 + MS['m100_rep2']['se_logQ'] ** 2))
GATES['G_m_decision_resolved'] = (adopted is not None)
REQ_B = ['G_m_run_inventory', 'G_m_all_Q_finite', 'G_m_two_rep_consistency'] + (['G_m_precision_each_run', 'G_m_decision_resolved'] if A10_MODE == 'official' else []); STATUS['M_SENSITIVITY_RESOLVED'] = all(GATES[k] for k in REQ_B)   # smoke N is too small for the precision gate: recorded, not required
atomic_json(dict(runs=MS, decision=M_DECISION, gates={k: GATES[k] for k in REQ_B}), os.path.join(CKPT, 'a10b_m_sensitivity.json')); print('m decision:', {k: v for k, v in M_DECISION.items() if k not in ('rule', 'cluster_ESS')}, '| status', STATUS['M_SENSITIVITY_RESOLVED'])


In [ ]:
# ---- A10c: ONE-POINT calibration PATHWAY prototype (Event B, 2D pseudo thresholds, no re-scan) with negative / positive controls and brute-force equality gate ----
m_use = M_DECISION['adopted_m'] if M_DECISION['adopted_m'] is not None else 100; dM, dI, cid, K = KEEP[m_use]['dM'], KEEP[m_use]['dI'], KEEP[m_use]['cid'], KEEP[m_use]['K']
(dP,), _, _ = (lambda o, c, g: ([o[(0, 'float32')]], c, g))(*generate(CFG['N_PSEUDO'], 1, (0,), [S_I])); pT1, pT2 = dP['T1'], dP['T2']
def hit_table_2d(T1, T2, cid, thr1, thr2):
    """H[k, p] = #{i in cluster k : T1_i <= thr1[p] and T2_i <= thr2[p]} via sort on T1 + cumulative per-cluster counts over the T2 condition (O(P * N) worst case, vectorised per pseudo)."""
    o = np.argsort(T1); T1s, T2s, cs = T1[o], T2[o], cid[o]; H = np.zeros((K, len(thr1)))
    for p in range(len(thr1)):
        n = np.searchsorted(T1s, thr1[p], side='right'); sel = T2s[:n] <= thr2[p]; H[:, p] = np.bincount(cs[:n][sel], minlength=K)
    return H
def brute_table(T1, T2, cid, thr1, thr2):
    H = np.zeros((K, len(thr1)))
    for p in range(len(thr1)): H[:, p] = np.bincount(cid[(T1 <= thr1[p]) & (T2 <= thr2[p])], minlength=K)
    return H
t = time.time(); HM, HI = hit_table_2d(dM['T1'], dM['T2'], cid, pT1, pT2), hit_table_2d(dI['T1'], dI['T2'], cid, pT1, pT2); t_tab = time.time() - t
nb = min(10, len(pT1)); GATES['G_cal_table_matches_bruteforce'] = bool(np.array_equal(HM[:, :nb], brute_table(dM['T1'], dM['T2'], cid, pT1[:nb], pT2[:nb])) and np.array_equal(HI[:, :nb], brute_table(dI['T1'], dI['T2'], cid, pT1[:nb], pT2[:nb])))
rb = np.random.default_rng(np.random.SeedSequence([MASTER_SEED, 11, STREAM['bootstrap'], 777])); Wc = np.zeros((CFG['B_BOOT'], K))
for b in range(CFG['B_BOOT']): Wc[b] = np.bincount(rb.integers(0, K, K), minlength=K)
def q_lower_ci(HM_, HI_):
    SM, SI = Wc @ HM_, Wc @ HI_; Qb = np.where((SI > 0) & (SM > 0), SM / np.maximum(SI, 1e-300), np.nan); return np.array([np.nanquantile(Qb[:, p], 0.025) if np.isfinite(Qb[:, p]).sum() > 10 else np.nan for p in range(HM_.shape[1])]), (HM_.sum(0) / len(dM['T1'])) / np.maximum(HI_.sum(0) / len(dI['T1']), 1e-300)
lo, Qp = q_lower_ci(HM, HI)
from scipy.stats import gaussian_kde
rf = rng_for('calibration', 5); fM = rf.choice(len(dM['T1']), CFG['N_FIT'], replace=False); fI = rf.choice(len(dI['T1']), CFG['N_FIT'], replace=False)
kM = gaussian_kde(np.vstack([dM['T1'][fM], dM['T2'][fM]])); kI = gaussian_kde(np.vstack([dI['T1'][fI], dI['T2'][fI]])); Dp = kM(np.vstack([pT1, pT2])) / np.maximum(kI(np.vstack([pT1, pT2])), 1e-300)
support = (lo >= 3) & (np.log(Dp) > 0); n_p = len(pT1); k_s = int(np.nansum(support))
# controls: negative = isotropic 'model' (S_I vs S_I clusters -> Q==1 exactly under CRN... use an independent iso replicate instead); positive = synthetic boosted model (T1 scaled by 0.5 -> Event B probability strongly increased)
(dN,), cidN, giN = (lambda o, c, g: ([o[(0, 'float32')]], c, g))(*generate(CFG['N'] if A10_MODE == 'official' else CFG['N'], m_use, (m_use, 9), [S_I])); HN = hit_table_2d(dN['T1'], dN['T2'], cidN, pT1, pT2); loN, QN = q_lower_ci(HN, HI)
kN = gaussian_kde(np.vstack([dN['T1'][fI], dN['T2'][fI]])); DN = kN(np.vstack([pT1, pT2])) / np.maximum(kI(np.vstack([pT1, pT2])), 1e-300); neg_support = (loN >= 3) & (np.log(DN) > 0)
HB = hit_table_2d(0.35 * dI['T1'], 0.35 * dI['T2'], cid, pT1, pT2); loB, QB = q_lower_ci(HB, HI); kB = gaussian_kde(np.vstack([0.35 * dI['T1'][fI], 0.35 * dI['T2'][fI]])); DB = kB(np.vstack([pT1, pT2])) / np.maximum(kI(np.vstack([pT1, pT2])), 1e-300); pos_support = (loB >= 3) & (np.log(DB) > 0)
HIo = hit_table_2d(dI['T1'], dI['T2'], cid, np.array([T1o]), np.array([T2o])); HBo = hit_table_2d(0.35 * dI['T1'], 0.35 * dI['T2'], cid, np.array([T1o]), np.array([T2o])); loBo, QBo = q_lower_ci(HBo, HIo)
# negative control: an independent isotropic replicate must not trigger support at the pseudo thresholds; positive control: at the OBSERVED thresholds (a tail event, P_iso ~ 4e-3) the boosted sample must trigger support with margin
GATES['G_cal_negative_control'] = bool(np.nanmean(neg_support) <= 0.05); GATES['G_cal_positive_control'] = bool(np.isfinite(loBo[0]) and loBo[0] >= 3 and QBo[0] >= 10)
CAL = dict(status_name='one_point_one_system_calibration_pathway_prototype', NOT='familywise global calibration (requires all families, both systems, family prior integration, n_pseudo >= 2000 or precision stopping; Phase B/C engine)',
           model_point='E7_b1_A x0(1), PR3-power-matched', m=m_use, n_pseudo=n_p, event='Event B with 2D pseudo thresholds (T1 <= T1_pseudo and T2 <= T2_pseudo)', method='sorted-T1 searchsorted + per-cluster counts; one shared cluster-resampling matrix (B x K) for every pseudo; Q lower CI = 2.5% percentile; D = KDE point estimate (fitting subsample separate from evaluation)',
           one_point_false_support_frequency=k_s / n_p, one_point_false_support_wilson=wilson(k_s, n_p), one_point_strong_raw_frequency=float(np.nanmean((lo >= 10) & (Dp > 1))),
           negative_control=dict(support_rate=float(np.nanmean(neg_support)), Q_median=float(np.nanmedian(QN))), positive_control=dict(support_rate_at_pseudo=float(np.nanmean(pos_support)), Q_median_at_pseudo=float(np.nanmedian(QB)), Q_at_observed=float(QBo[0]), Q_lowerCI_at_observed=float(loBo[0]), note='at central pseudo thresholds the ratio is capped by 1/P_iso, so the control is evaluated at the observed (tail) thresholds', construction='isotropic sample with T1 and T2 scaled by 0.35 (synthetic boosted Event B), same clusters'),
           Q_pseudo_median=float(np.nanmedian(Qp)), pseudo_T1_q=np.quantile(pT1, [0.05, 0.5, 0.95]).tolist(), pseudo_T2_q=np.quantile(pT2, [0.05, 0.5, 0.95]).tolist(), table_seconds=t_tab)
REQ_C = ['G_cal_table_matches_bruteforce', 'G_cal_negative_control', 'G_cal_positive_control']; STATUS['CALIBRATION_PATH_VALID'] = all(GATES[k] for k in REQ_C)
np.savez_compressed(os.path.join(CKPT, 'a10c_pathway.npz'), pseudo_T1=pT1, pseudo_T2=pT2, Q=Qp, Q_lowerCI=lo, D=Dp, Q_neg=QN, Q_neg_lowerCI=loN, Q_pos=QB, Q_pos_lowerCI=loB); atomic_json(dict(CAL=CAL, gates={k: GATES[k] for k in REQ_C}), os.path.join(CKPT, 'a10c_pathway.json'))
print('A10c pathway:', {k: CAL[k] for k in ('one_point_false_support_frequency', 'negative_control', 'positive_control', 'Q_pseudo_median')}, '| status', STATUS['CALIBRATION_PATH_VALID'])


In [ ]:
# ---- A10d: W2 estimator on real engine output: 3-position max-pairwise trigger (mock positions from the A11 cache), max-pairwise null with disjoint cluster blocks, MC p, B / n_sub / seed sensitivity ----
def white(d): return (np.column_stack([d['T1'], d['T2']]) - mu_c) @ Sih.T
def w2_exact(a, b):
    Mc = ot.dist(a, b, metric='sqeuclidean'); v, lg = ot.emd2(np.full(len(a), 1 / len(a)), np.full(len(b), 1 / len(b)), Mc, numItermax=1_000_000, log=True); return float(np.sqrt(v)), int(bool(lg.get('warning')))
m_w2 = m_use; S_POS = [psqrt(matched(C_CT[k])[0])[0] for k in ('cov_P1', 'cov_P2', 'cov_P3')]
t = time.time(); pos_out, pos_cid, gi = generate(CFG['N_W2_POOL'], m_w2, (3,), S_POS); TwP = [white(pos_out[(s, 'float32')]) for s in range(3)]; Kp = gi['K']
iso_out, iso_cid, gi2 = generate(3 * CFG['N_W2_POOL'], m_w2, (4,), [S_I]); TwI = white(iso_out[(0, 'float32')]); Ki = gi2['K']; t_gen = time.time() - t
def take(Tw, cid, clusters): return Tw[np.isin(cid, clusters)]
W2 = {}; t = time.time(); warn = 0
for n_sub in CFG['N_SUB_W2']:
    kc = n_sub // m_w2; rw = rng_for('w2', n_sub)
    # observed: max over the 3 position pairs (same cluster subset for the three positions -> CRN)
    obs = {}
    for seed in range(3):
        kk = rw.choice(Kp, kc, replace=False); A = [take(TwP[s], pos_cid, kk) for s in range(3)]; pw = {}
        for a in range(3):
            for b in range(a + 1, 3): v, w = w2_exact(A[a], A[b]); pw[f'P{a+1}P{b+1}'] = v; warn += w
        obs[f'seed{seed}'] = dict(pairwise=pw, W2_max=max(pw.values()))
    # null: each replicate draws THREE disjoint cluster blocks from the isotropic pool (no reuse within a replicate; blocks advance across replicates)
    null = []; perm = rw.permutation(Ki); ptr = 0
    for b in range(CFG['B_NULL']):
        if ptr + 3 * kc > Ki: perm = rw.permutation(Ki); ptr = 0
        blocks = [perm[ptr + j * kc: ptr + (j + 1) * kc] for j in range(3)]; ptr += 3 * kc; S3 = [take(TwI, iso_cid, bl) for bl in blocks]; vals = []
        for a in range(3):
            for c in range(a + 1, 3): v, w = w2_exact(S3[a], S3[c]); vals.append(v); warn += w
        null.append(max(vals))
    null = np.array(null); W2[str(n_sub)] = dict(n_sub=n_sub, clusters_per_sample=kc, observed=obs, null_q99_method_higher=float(np.quantile(null, 0.99, method='higher')), null_q95=float(np.quantile(null, 0.95, method='higher')), null_median=float(np.median(null)), null_max=float(null.max()),
                                                 mc_p_seed0=float((1 + np.sum(null >= obs['seed0']['W2_max'])) / (len(null) + 1)), mc_p_all_seeds=[float((1 + np.sum(null >= obs[f'seed{s}']['W2_max'])) / (len(null) + 1)) for s in range(3)], null_values=null.tolist())
    print(f"W2 n_sub={n_sub}: obs W2_max {[round(obs[f'seed{s}']['W2_max'],4) for s in range(3)]} | null q99 {W2[str(n_sub)]['null_q99_method_higher']:.4f} median {W2[str(n_sub)]['null_median']:.4f} | MC p {W2[str(n_sub)]['mc_p_all_seeds']} ({time.time()-t:.0f}s)")
GATES['G_w2_solver_clean'] = (warn == 0 and all(np.all(np.isfinite(W2[k]['null_values'])) for k in W2)); GATES['G_w2_nsub_inventory'] = (set(W2) == {str(n) for n in CFG['N_SUB_W2']})
REC['w2'] = dict(estimator='exact 2D W2 (POT emd2, sqeuclidean, sqrt), uniform weights', pot_version=ot.__version__, whitening='calibration sample mean/cov', positions_mock={k: X0[k] for k in X0}, positions_note='mock 3-position set from the A11 cache (E7_b1_A base, E7_b2_A base, y-shift 0.12); NOT the registered p^(3) points',
                trigger='W2_max over 3 pairs vs null of max over 3 disjoint isotropic samples; threshold quantile method=higher; MC p = (1 + #null >= obs)/(B+1)', m=m_w2, B_null=CFG['B_NULL'], results={k: {kk: vv for kk, vv in v.items() if kk != 'null_values'} for k, v in W2.items()}, generation_seconds=t_gen)
REQ_D = ['G_w2_solver_clean', 'G_w2_nsub_inventory']; STATUS['W2_ESTIMATOR_VALID'] = all(GATES[k] for k in REQ_D)
np.savez_compressed(os.path.join(CKPT, 'a10d_w2.npz'), **{f'null_nsub{k}': np.array(v['null_values']) for k, v in W2.items()}); atomic_json(dict(w2=REC['w2'], gates={k: GATES[k] for k in REQ_D}), os.path.join(CKPT, 'a10d_w2.json')); print('A10d status', STATUS['W2_ESTIMATOR_VALID'])


In [ ]:
# ---- A10e: provenance (component statuses -> A10_VALID), output SHA, atomic write, final assert ----
REQUIRED = REQ_A + REQ_B + REQ_C + REQ_D; INVENTORY = set(REQ_A) | {'G_m_run_inventory', 'G_m_all_Q_finite', 'G_m_two_rep_consistency', 'G_m_precision_each_run', 'G_m_decision_resolved'} | set(REQ_C) | set(REQ_D) | {'G_gate_inventory_exact'}
GATES['G_gate_inventory_exact'] = (set(GATES) | {'G_gate_inventory_exact'} == INVENTORY); REQUIRED = REQUIRED + ['G_gate_inventory_exact']
ALL = all(GATES[k] for k in REQUIRED) and all(STATUS.values()); status = ('A10_VALID' if A10_MODE == 'official' else 'SMOKE_PASS') if ALL else 'FAILED'
outputs = {f: sha(os.path.join(CKPT, f)) for f in sorted(os.listdir(CKPT)) if not f.endswith('.tmp')}
prov = dict(notebook=f'Step1 Phase A-10 v1.1 [{A10_MODE}]', status=status, component_status=STATUS, timestamp=time.strftime('%Y-%m-%dT%H:%M:%S'), mode=A10_MODE, config=CFG, master_seed=MASTER_SEED, streams=STREAM, production_path=PROD, threads_live=THREADS_LIVE,
            notebook_identity=dict(basename=NB_BASENAME, head_copy=NB_HEAD), repo=dict(commit=MT_COMMIT, git_calls=GIT_CALLS), environment=dict(versions=VERS, expected=EXPECTED_VERS, mismatch=VERS_MISMATCH, platform=platform.platform(), in_colab=IN_COLAB),
            assets=ASSET_SHA, module_sha=EXP_MOD, quadrature=DIAG['quadrature_sha256'], event='Event B = {T1 <= T1_obs and T2 <= T2_obs} (A5-frozen primary event); central-band E_sel of v1.0 is superseded', gates=GATES, required_gates=REQUIRED,
            covariance=REC['covariance'], calibration=REC['calibration'], m_sensitivity=dict(runs=MS, decision=M_DECISION), calibration_pathway=CAL, w2=REC['w2'], outputs=outputs, superseded=dict(v1_0='central-band event + float64 selection + chunk 10000; Q~1.08-1.11, m=100, FWFSR 0/200 withdrawn'))
atomic_json(prov, os.path.join(OUT, 'a10_provenance.json')); print('STATUS =', status, '| gates', sum(GATES[k] for k in REQUIRED), '/', len(REQUIRED), '| components', STATUS, '| provenance SHA', sha(os.path.join(OUT, 'a10_provenance.json'))[:16])
assert status != 'FAILED', {k: GATES[k] for k in REQUIRED if not GATES[k]}


## 実行手順
1. **smoke**（fresh runtime・数分）：先頭に `A10_MODE='smoke'` セルを追加 → Run all → `STATUS = SMOKE_PASS`（4 component 全 True）。smoke では版 gate・m 精度 gate・m 判定 gate は記録のみ（N が小さいため），official で required。
2. **official**（fresh runtime・CPU で可・1〜1.5 時間；W₂ null が支配的）：純正ノートを Run all → `STATUS = A10_VALID`。official では期待版（Python 3.13.15・NumPy 2.1.3・SciPy 1.16.3・healpy 1.20.0・POT 0.9.7.post1）との不一致で停止する。
3. 返送：`runs_step1_phaseA/a10_v1.1_{smoke,official}/a10_provenance.json` と `checkpoints/`（a10a〜a10d の npz/json）・全セル出力。
